<a href="https://colab.research.google.com/github/Kjrob/Demographics-Research/blob/main/ESW_Research.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
df = pd.read_excel("/content/GreenVoicesDiverseChoices.xlsx", skiprows=[1,2])
df = df.dropna(how="all")
df = df.reset_index(drop=True)

FileNotFoundError: [Errno 2] No such file or directory: '/content/GreenVoicesDiverseChoices.xlsx'

In [ ]:
demographics = ["QID4", "QID20", "QID7", "QID22", "QID23", "QID28", "QID34"]
mc_questions = [col for col in [str(c) for c in df.columns] if col.startswith("Q") and col not in demographics]

In [ ]:
for col in demographics + mc_questions:
    print(f"\n{col}")
    print(df[col].value_counts())

In [ ]:
# --------------------------
# Full Survey Analysis Script
# --------------------------

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import chi2_contingency
import os

# -------- STEP 0: File path --------
# Replace with the actual path to your CSV
csv_path = "/content/GreenVoicesDiverseChoices.xlsx"

# Check current working directory
print("Current directory:", os.getcwd())
print("Files here:", os.listdir(os.path.dirname(csv_path)))

# -------- STEP 1: Load & clean data --------
df = pd.read_excel(csv_path, skiprows=[1,2])  # skip extra header rows
df = df.dropna(how="all")
df = df.reset_index(drop=True)
df.columns = df.columns.astype(str) # Convert all column names to strings

# -------- STEP 2: Define demographics and questions --------
demographics = ["Q1", "Q2", "Q3"]  # Example: age, role, gender
mc_questions = [col for col in df.columns if col.startswith("Q") and col not in demographics]
text_questions = [col for col in df.columns if "_TEXT" in col]  # free response columns

# -------- STEP 3: Basic sanity check --------
print("\n=== Value counts for demographics ===")
for demo in demographics:
    print(f"\n{demo} distribution:\n{df[demo].value_counts()}")

# -------- STEP 4: Loop through MC questions vs demographics --------
print("\n=== Multiple Choice Analysis ===")

for demo in demographics:
    for q in mc_questions:
        # Skip if demo or question is empty
        if demo not in df.columns or q not in df.columns:
            continue

        # Crosstab
        ct = pd.crosstab(df[demo], df[q])
        if ct.shape[0] < 2 or ct.shape[1] < 2:
            continue  # nothing to test

        # Normalize by row
        ct_norm = pd.crosstab(df[demo], df[q], normalize="index")

        # Chi-square test
        chi2, p, _, _ = chi2_contingency(ct)

        # Print results
        print(f"\n--- {demo} vs {q} ---")
        print(ct_norm)
        print(f"p-value: {p:.4f}")
        if p < 0.05:
            print("→ Significant relationship")
        else:
            print("→ Not significant")

        # Heatmap
        plt.figure(figsize=(10,6))
        sns.heatmap(ct_norm, annot=True, fmt=".2f", cmap="Blues")
        plt.title(f"{demo} vs {q} (Proportion)")
        plt.ylabel("Demographic Group")
        plt.xlabel("Response Option")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

# -------- STEP 5: Free response keyword analysis --------
print("\n=== Free Response Analysis ===")

# Example: define keywords for one free response question
keywords = ["cost", "expensive", "price", "time", "interest"]  # add your own
for text_col in text_questions:
    for kw in keywords:
        feature_col = f"{text_col}_mentions_{kw}"
        df[feature_col] = df[text_col].str.contains(kw, case=False, na=False)

        # Crosstab with a demographic, e.g., Age
        demo = "Q1"
        ct = pd.crosstab(df[demo], df[feature_col], normalize="index")
        print(f"\n{demo} vs mentions '{kw}' in {text_col}:")
        print(ct)

        # Heatmap
        plt.figure(figsize=(8,4))
        sns.heatmap(ct, annot=True, fmt=".2f", cmap="Greens")
        plt.title(f"{demo} vs mentions '{kw}'")
        plt.ylabel("Demographic Group")
        plt.xlabel(f"Mentions '{kw}'")
        plt.tight_layout()
        plt.show()

# -------- STEP 6: Save normalized results (optional) --------
# ct_norm.to_csv("normalized_results.csv")